# Milestone 2: Architecture Logic
**Course:** KD34403 Machine Learning for Data Science  
**University:** Universiti Malaysia Sabah  
**Group:** 10  
**Author:** Julaika Ang BI23110160

---
### Objective
Explain and justify the machine learning model architecture chosen for predicting IMDB movie ratings. This milestone covers:
- Why we chose Gradient Boosting Regressor
- How the full pipeline is structured
- Why each component works for this task

## 1. Problem Definition
This is a **supervised regression** task.

| Property | Detail |
|----------|--------|
| Task type | Regression (predict a continuous value) |
| Target variable | `IMDB_Rating` (1.0 – 10.0) |
| Input features | Genre, Runtime, Votes, Year, Gross, Certificate, Meta_score |
| Evaluation metrics | RMSE, MAE, R² |

We predict a **continuous rating score**, not a category — so regression models are appropriate, not classifiers.

## 2. Model Choice — Gradient Boosting Regressor

We selected **Gradient Boosting Regressor** (from scikit-learn) for the following reasons:

### Why not Linear Regression?
Linear Regression assumes a straight-line relationship between features and the target. IMDB ratings are influenced by complex, non-linear interactions (e.g. a Drama with high votes may score very differently from an Action film with high votes). Linear Regression cannot capture this.

### Why not a Neural Network?
Our dataset has only ~1,000 records. Neural networks require large amounts of data to generalise well. On small datasets they tend to overfit and are harder to tune without GPU resources.

### Why Gradient Boosting Regressor? ✅
- Handles **non-linear relationships** between features and ratings
- Works well on **small-to-medium tabular datasets**
- Built-in handling of feature interactions (no manual feature engineering needed)
- More resistant to outliers than simple decision trees
- Produces interpretable **feature importance scores**
- Can run on a standard laptop without GPU

## 3. Full Pipeline Architecture

```
Raw IMDB Dataset (CSV)
        │
        ▼
┌─────────────────────────┐
│   Step 1: Data Loading  │  pd.read_csv()
└─────────────────────────┘
        │
        ▼
┌─────────────────────────┐
│  Step 2: Data Cleaning  │  Drop nulls, fix dtypes, extract Runtime_min
└─────────────────────────┘
        │
        ▼
┌─────────────────────────┐
│  Step 3: Feature Select │  X = [Year, Runtime, Certificate, Genre,
│                         │       Meta_score, Votes, Gross]
│                         │  y = IMDB_Rating
└─────────────────────────┘
        │
        ▼
┌─────────────────────────┐
│  Step 4: Train/Val/Test │  70% Train | 15% Validation | 15% Test
│          Split          │  (random_state=42 for reproducibility)
└─────────────────────────┘
        │
        ▼
┌─────────────────────────┐
│  Step 5: Preprocessing  │  Numeric  → StandardScaler
│  (ColumnTransformer)    │  Categorical → OneHotEncoder
└─────────────────────────┘
        │
        ▼
┌─────────────────────────┐
│  Step 6: Model Training │  GradientBoostingRegressor
│                         │  n_estimators=300, lr=0.05, max_depth=3
└─────────────────────────┘
        │
        ▼
┌─────────────────────────┐
│  Step 7: Evaluation     │  RMSE, MAE, R² on test set
│                         │  Actual vs Predicted scatter plot
└─────────────────────────┘
```

## 4. Feature Justification

| Feature | Type | Why included |
|---------|------|-------------|
| `Released_Year` | Numeric | Older films may have rating bias from nostalgia |
| `Runtime_min` | Numeric | Longer films tend to be more serious/acclaimed |
| `Meta_score` | Numeric | Strong correlation with IMDB Rating (critic vs audience) |
| `No_of_Votes` | Numeric | More votes = more reliable rating signal |
| `Gross` | Numeric | Box office success loosely correlates with popularity |
| `Certificate` | Categorical | Age rating influences audience demographic |
| `Genre` | Categorical | Genre strongly affects expected rating range |

### Preprocessing choices
- **StandardScaler** on numeric features — Gradient Boosting is not sensitive to scale, but scaling helps convergence and is good practice
- **OneHotEncoder** on categorical features — converts Genre and Certificate into binary columns the model can use
- Preprocessor is **fit only on training data** to prevent data leakage into validation/test sets

## 5. Hyperparameter Choices

| Hyperparameter | Value | Reason |
|---------------|-------|--------|
| `n_estimators` | 300 | Selected by tracking validation RMSE across [10,20,50,100,150,200,300] stages |
| `learning_rate` | 0.05 | Low learning rate with more trees — better generalisation than high lr + few trees |
| `max_depth` | 3 | Shallow trees reduce overfitting; Gradient Boosting works best with weak learners |
| `random_state` | 42 | Ensures results are reproducible across machines |

## 6. Why It Works — The Gradient Boosting Mechanism

Gradient Boosting builds an ensemble of **weak decision trees** sequentially. Each new tree corrects the **residual errors** (mistakes) of the previous trees.

**Step-by-step:**
1. Start with a simple prediction (the mean IMDB rating)
2. Calculate the error (residual) between prediction and actual rating
3. Train a shallow decision tree to predict that residual
4. Add the tree's correction to the running prediction (scaled by `learning_rate`)
5. Repeat for `n_estimators` rounds

**Why this suits IMDB ratings:**
- Ratings are influenced by many interacting factors (genre × votes × year) — boosting naturally captures these interactions across trees
- The sequential correction process means later trees focus on the *hardest* movies to predict (unusual films, niche genres)
- With `max_depth=3`, each tree only makes simple splits — the power comes from combining 300 of them

## 7. Summary

| Decision | Choice | Alternative considered |
|----------|--------|----------------------|
| Model type | Gradient Boosting Regressor | Linear Regression, Neural Network |
| Categorical encoding | OneHotEncoder | LabelEncoder |
| Numeric scaling | StandardScaler | MinMaxScaler |
| Train/val/test split | 70/15/15 | 80/20 |
| Hyperparameter selection | Validation RMSE curve | Grid search |

The Gradient Boosting Regressor was chosen because it best balances **predictive power**, **interpretability**, and **computational efficiency** for a small tabular dataset like IMDB Top 1000 movies.

## 8. Code — Feature and Pipeline Definition

In [ ]:
# This cell shows the feature selection and pipeline structure
# Full training code is in Milestone 3

import pandas as pd

# Define feature groups
numeric_features = [
    "Released_Year",   # Year the movie was released
    "Runtime_min",     # Runtime in minutes (extracted from text)
    "Meta_score",      # Metacritic score (critic rating)
    "No_of_Votes",     # Number of user votes on IMDB
    "Gross"            # Box office gross revenue
]

categorical_features = [
    "Certificate",     # Age rating (U, UA, A, etc.)
    "Genre"            # Movie genre (Drama, Action, etc.)
]

target_variable = "IMDB_Rating"  # Continuous value 1.0 – 10.0

print("Input features (numeric)    :", numeric_features)
print("Input features (categorical):", categorical_features)
print("Target variable             :", target_variable)
print("Total input features        :", len(numeric_features) + len(categorical_features))
print("Model                       : GradientBoostingRegressor")
print("Task type                   : Regression (predict continuous rating)")

In [ ]:
# Show the ColumnTransformer pipeline structure
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline

# Build preprocessing step
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(transformers=[
    ("numeric",      StandardScaler(), numeric_features),
    ("categorical",  ohe,              categorical_features)
])

# Build full pipeline: preprocessing + model
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model",        GradientBoostingRegressor(
                         n_estimators=300,
                         learning_rate=0.05,
                         max_depth=3,
                         random_state=42
                     ))
])

print("Full ML Pipeline:")
print(pipeline)